# VCPI Drug-seq Hackathon

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Click the 🔑 key icon (left sidebar) → add secret `TVC_TOKEN` → enable notebook access
3. Run All (Ctrl+F9)

This notebook clones the repo, installs deps, loads data from GCS, and is ready to train.

In [ ]:
# ── Cell 1: Clone / pull latest code ─────────────────────────────────────────
import os

REPO_URL    = 'https://github.com/mayafrommiami/vcpi-client.git'
REPO_BRANCH = 'hackathon'
REPO_DIR    = '/content/vcpi-hack'

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull origin {REPO_BRANCH}
else:
    !git clone --branch {REPO_BRANCH} --single-branch {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f'Working dir: {os.getcwd()}')
!git log --oneline -5

In [ ]:
# ── Cell 2: Install packages ──────────────────────────────────────────────────
!pip install \
    git+https://github.com/virtualcell-vcpi/vcpi-client.git \
    git+https://github.com/virtualcell-vcpi/vcpi-prediction-contest-2026.git \
    google-cloud-storage \
    polars pyarrow \
    rdkit seaborn scikit-learn scipy \
    torch torchvision \
    --quiet
print('Done.')

In [ ]:
# ── Cell 3: Auth — TVC_TOKEN only (GCS is public, no auth needed) ─────────────
import os

# Try Colab Secrets first; fall back to manual paste
try:
    from google.colab import userdata
    token = userdata.get('TVC_TOKEN')
except Exception:
    token = None

if not token:
    import getpass
    token = getpass.getpass('Paste your TVC_TOKEN and press Enter: ')

os.environ['TVC_TOKEN']  = token
os.environ['GCS_BUCKET'] = 'vcpi-drugseq-2026'
assert os.environ['TVC_TOKEN'], 'TVC_TOKEN is empty — check your token'
print('TVC_TOKEN set. (GCS bucket is public — no GCS auth needed)')

In [ ]:
# ── Cell 4: Check GPU ─────────────────────────────────────────────────────────
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE')
print('CUDA:', torch.version.cuda)
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

In [ ]:
# ── Cell 5: GCS loader (anonymous — bucket is public) ────────────────────────
import io
import polars as pl
from google.cloud import storage

BUCKET = os.environ['GCS_BUCKET']

# anonymous_credentials = no auth required (bucket is publicly readable)
_client = storage.Client.create_anonymous_client()

def load_parquet(blob_path: str) -> pl.DataFrame:
    buf = io.BytesIO()
    _client.bucket(BUCKET).blob(blob_path).download_to_file(buf)
    buf.seek(0)
    return pl.read_parquet(buf)

print(f'GCS loader ready  gs://{BUCKET}  (public bucket, no credentials needed)')

In [ ]:
# ── Cell 6: Load training data ────────────────────────────────────────────────
# Loads one dataset at a time (Colab has ~12GB RAM)
# For full training, load all 3 and concat expression

DATASETS = ['tvc-bhr-009', 'tvc-kdl-010', 'tvc-qnu-012']

counts_009 = load_parquet('data/tvc-bhr-009/counts.parquet')
meta_009   = load_parquet('data/tvc-bhr-009/metadata.parquet')
chem_009   = load_parquet('data/tvc-bhr-009/chemistry.parquet')
print(f'tvc-bhr-009  counts: {counts_009.shape}  meta: {meta_009.shape}  chem: {chem_009.shape}')

# Uncomment for additional plates:
# counts_010 = load_parquet('data/tvc-kdl-010/counts.parquet')
# meta_010   = load_parquet('data/tvc-kdl-010/metadata.parquet')
# chem_010   = load_parquet('data/tvc-kdl-010/chemistry.parquet')

# counts_012 = load_parquet('data/tvc-qnu-012/counts.parquet')
# meta_012   = load_parquet('data/tvc-qnu-012/metadata.parquet')
# chem_012   = load_parquet('data/tvc-qnu-012/chemistry.parquet')

In [ ]:
# ── Cell 7: Contest utilities + baseline ──────────────────────────────────────
import sys
sys.path.insert(0, REPO_DIR)

from vcpi_prediction_contest import (
    counts_to_expression, load_gene_filter, load_test_compounds,
    score_compounds, predict_mu_all_train,
)

gene_filter    = load_gene_filter()
test_compounds = load_test_compounds()
print(f'Gene filter: {len(gene_filter)} genes | Test compounds: {len(test_compounds)}')

# Normalize counts → log2(CPM+1), aggregate by compound
expr_009 = counts_to_expression(counts_009.to_pandas(), meta_009.to_pandas())
print(f'Expression (long): {expr_009.shape}')

# Baseline wMSE benchmark
baseline = predict_mu_all_train(
    truth_train=expr_009,
    test_compounds=test_compounds['compound'].tolist(),
    gene_filter=gene_filter,
)
print('Baseline prediction ready (target to beat: 0.507 wMSE)')

In [ ]:
# ── Cell 8: Build dataset + train MLP ────────────────────────────────────────
# Uses models/data_loader.py and models/mlp.py from the cloned repo

from models.data_loader import build_dataset, get_dataloaders
from models.mlp import FingerprintMLP, WeightedMSELoss, build_loss_weights

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training on: {DEVICE}')

# Build dataset from GCS (downloads + computes fingerprints, ~5 min)
dataset   = build_dataset()   # uses all 3 plates by default
train_dl, val_dl = get_dataloaders(dataset, val_frac=0.1, batch_size=64)

# Model
n_genes = len(dataset.gene_ids)
model   = FingerprintMLP(fp_dim=2048, n_genes=n_genes).to(DEVICE)
print(f'Model params: {sum(p.numel() for p in model.parameters()):,}')

# Loss: weighted by per-gene variance across compounds
weights = build_loss_weights(dataset.expressions).to(DEVICE)
loss_fn = WeightedMSELoss(weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

In [ ]:
# ── Cell 9: Training loop ─────────────────────────────────────────────────────
from torch.optim.lr_scheduler import CosineAnnealingLR

EPOCHS    = 100
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

train_losses, val_losses = [], []

for epoch in range(1, EPOCHS + 1):
    # ── train ──
    model.train()
    t_loss = 0.0
    for fp, target in train_dl:
        fp, target = fp.to(DEVICE), target.to(DEVICE)
        optimizer.zero_grad()
        pred = model(fp)
        loss = loss_fn(pred, target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        t_loss += loss.item()
    scheduler.step()

    # ── val ──
    model.eval()
    v_loss = 0.0
    with torch.no_grad():
        for fp, target in val_dl:
            fp, target = fp.to(DEVICE), target.to(DEVICE)
            v_loss += loss_fn(model(fp), target).item()

    train_losses.append(t_loss / len(train_dl))
    val_losses.append(v_loss / len(val_dl))

    if epoch % 10 == 0:
        print(f'Epoch {epoch:3d}/{EPOCHS}  train={train_losses[-1]:.4f}  val={val_losses[-1]:.4f}  lr={scheduler.get_last_lr()[0]:.2e}')

print('Training complete.')

In [ ]:
# ── Cell 10: Generate submission ──────────────────────────────────────────────
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from models.data_loader import smiles_to_fingerprint

model.eval()
per_gene_mean = torch.tensor(dataset.per_gene_mean, device=DEVICE)

rows = []
with torch.no_grad():
    for _, row in test_compounds.iterrows():
        fp = smiles_to_fingerprint(row['smiles'])
        if fp is None:
            # Fallback: per-gene mean for invalid SMILES
            pred = per_gene_mean.cpu().numpy()
        else:
            fp_t  = torch.tensor(fp).unsqueeze(0).to(DEVICE)
            residual = model(fp_t).squeeze(0)
            pred  = (per_gene_mean + residual).clamp(min=0).cpu().numpy()

        for gene, val in zip(dataset.gene_ids, pred):
            rows.append({'compound': row['compound'], 'gene_id': gene, 'predicted_expression': float(val)})

submission = pd.DataFrame(rows)
submission.to_parquet('submission.parquet', index=False)
print(f'Submission: {submission.shape}  ({submission["compound"].nunique()} compounds × {submission["gene_id"].nunique()} genes)')
print(submission.head())